In [1]:
import torch
import math

In [2]:
Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

K = torch.tensor([
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

V = torch.tensor([
    [10.0, 0.0],
    [0.0, 20.0],
    [5.0, 5.0]
])

Compute scores.

In [3]:
scores = Q @ K.T

print(scores)

tensor([[1., 1., 0.],
        [0., 1., 1.],
        [1., 2., 1.]])


Create Mask

In [4]:
seq_len = scores.shape[0]

mask = torch.triu(
    torch.ones(seq_len, seq_len),
    diagonal=1
)

What Does triu Mean?

Upper triangle.

Everything above the diagonal becomes:

future positions

In [5]:
print(mask)

tensor([[0., 1., 1.],
        [0., 0., 1.],
        [0., 0., 0.]])


Convert Future Positions to -inf

In [6]:
mask = mask.masked_fill(
    mask == 1,
    float("-inf")
)

print(mask)

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])


Apply Mask

In [7]:
masked_scores = scores + mask

print(masked_scores)

tensor([[1., -inf, -inf],
        [0., 1., -inf],
        [1., 2., 1.]])


Softmax

In [9]:
weights = torch.softmax(
    masked_scores,
    dim=-1
)

print(weights)

tensor([[1.0000, 0.0000, 0.0000],
        [0.2689, 0.7311, 0.0000],
        [0.2119, 0.5761, 0.2119]])


Compute Output

In [10]:
output = weights @ V

print(output)

tensor([[10.0000,  0.0000],
        [ 2.6894, 14.6212],
        [ 3.1791, 12.5820]])


Wrap Everything Into GPT Attention

In [11]:
def causal_attention(Q, K, V):

    scores = Q @ K.T

    scores = scores / math.sqrt(
        K.shape[-1]
    )

    seq_len = scores.shape[0]

    mask = torch.triu(
        torch.ones(seq_len, seq_len),
        diagonal=1
    )

    mask = mask.masked_fill(
        mask == 1,
        float("-inf")
    )

    scores = scores + mask

    weights = torch.softmax(
        scores,
        dim=-1
    )

    output = weights @ V

    return output, weights